# 02 — Compute Custom Growth Signals

The Vivameda dataset includes four pre-computed signal flags: `early_scaling_flag`, `contraction_flag`, `recovery_signal_flag`, and `growth_acceleration_flag`. These are deterministic functions of the raw growth fields.

Many ML practitioners will want to compute their own signals tailored to specific use cases. This notebook shows:

1. The exact definition of each built-in flag
2. How to verify them from raw fields
3. How to design your own signal definitions

Dataset: [Vivameda/longitudinal_503companies_1950_2020](https://huggingface.co/datasets/Vivameda/longitudinal_503companies_1950_2020)


## Setup

In [ ]:
import pandas as pd
import numpy as np

HF_URL = "https://huggingface.co/datasets/Vivameda/longitudinal_503companies_1950_2020/resolve/main/vivameda_longitudinal_sample_503companies_1950_2020.csv"
df = pd.read_csv(HF_URL)
print(f"Loaded {len(df):,} rows")

## The four built-in signal flags

Each flag is computed from `growth_rate_yoy`, `prev_growth_rate_yoy`, and `headcount_observed`. NULL when prerequisites aren't available (typically the first observed year of a company).

```
early_scaling_flag      = 1 if growth_rate_yoy > 30 AND headcount BETWEEN 50 AND 500 AND prev_growth_rate_yoy > 10
contraction_flag        = 1 if growth_rate_yoy < -5 AND prev_growth_rate_yoy > 0
recovery_signal_flag    = 1 if growth_rate_yoy > 5 AND prev_growth_rate_yoy < -5
growth_acceleration_flag = 1 if growth_rate_yoy > prev_growth_rate_yoy AND growth_rate_yoy > 10
```

These definitions are deterministic and reproducible. Anyone can verify them from the raw fields.

In [ ]:
# Verify early_scaling_flag from raw fields
def compute_early_scaling(row):
    if pd.isna(row['growth_rate_yoy']) or pd.isna(row['prev_growth_rate_yoy']):
        return np.nan
    return int(
        row['growth_rate_yoy'] > 30
        and 50 <= row['headcount_observed'] <= 500
        and row['prev_growth_rate_yoy'] > 10
    )

df['early_scaling_check'] = df.apply(compute_early_scaling, axis=1)

# Compare to the pre-computed flag
match = (df['early_scaling_flag'] == df['early_scaling_check']).sum()
total = df['early_scaling_flag'].notna().sum()
print(f"Match rate: {match}/{total} ({match/total*100:.1f}%)")

## How often does each signal fire?

Useful to understand baseline rates before designing custom signals.

In [ ]:
flags = ['early_scaling_flag', 'contraction_flag', 'recovery_signal_flag', 'growth_acceleration_flag']
for f in flags:
    fired = (df[f] == 1).sum()
    total = df[f].notna().sum()
    rate = fired / total * 100 if total else 0
    print(f"{f:30s}  fires in {fired:5,} / {total:6,} rows ({rate:.1f}%)")

## Designing a custom signal

Suppose we want to flag "post-recession recoveries" — companies that contracted significantly in a known recession year (2008 or 2009) and then grew back the following year.

This is exactly the kind of regime-aware signal that's hard to extract from text-trained models but trivial from longitudinal panel data.

In [ ]:
def post_recession_recovery(g):
    """
    Flag company-years where:
    - The previous year was 2008 or 2009 (recession)
    - That year saw growth_rate_yoy < -10
    - The current year shows growth_rate_yoy > 5
    """
    g = g.sort_values('year').copy()
    g['prev_year'] = g['year'].shift(1)
    g['prev_growth'] = g['growth_rate_yoy'].shift(1)
    g['recession_recovery'] = (
        g['prev_year'].isin([2008, 2009])
        & (g['prev_growth'] < -10)
        & (g['growth_rate_yoy'] > 5)
    ).astype(int)
    return g

df_with_signal = df.groupby('company_id', group_keys=False).apply(post_recession_recovery)
recoveries = df_with_signal[df_with_signal['recession_recovery'] == 1]
print(f"Post-recession recoveries identified: {len(recoveries)}")
print()
print("Sample of recovering companies:")
print(recoveries[['company_name', 'year', 'industry', 'prev_growth', 'growth_rate_yoy']].head(15))

## Industry-conditional signals

Another useful pattern: signals that fire only within specific industry contexts. The dataset's 75-category taxonomy makes this easy.

In [ ]:
# Tech-sector hypergrowth: high growth specifically in software/internet/IT
tech_industries = ['Computer Software', 'Internet', 'Information Technology and Services']
df['tech_hypergrowth'] = (
    (df['industry'].isin(tech_industries))
    & (df['growth_rate_yoy'] > 50)
    & (df['headcount_observed'] >= 100)
).astype(int)

print(f"Tech hypergrowth events: {df['tech_hypergrowth'].sum()}")
print()
events = df[df['tech_hypergrowth'] == 1].sort_values('year')
print(events[['company_name', 'year', 'headcount_observed', 'growth_rate_yoy']].head(15))

## Why this matters for AI training

Pre-computed flags are convenient for evaluation, but they encode specific definitions that may not fit your use case.

The raw fields (`growth_rate_yoy`, `prev_growth_rate_yoy`, `headcount_observed`, `net_headcount_change`, `avg_tenure_years`) are always available, including in rows where the flags themselves are NULL.

This is intentional: the dataset is a substrate. You define the signals.

## Continue exploring

Try designing signals for:

- **Capability shifts** — using `top_capability_1` transitions across years to detect strategic pivots
- **Role mix evolution** — using `primary_role_bucket` and `distinct_role_buckets` to detect organizational restructuring
- **Tenure regime changes** — using `tenure_bucket` shifts to detect generational turnover
- **Multi-year compound signals** — chaining single-year signals across 3-5 year windows

The 70-year temporal coverage means models trained on this substrate see organizational dynamics across multiple economic regimes — postwar expansion, stagflation, the dot-com boom and bust, the 2008 financial crisis, the platform era, and the late-cycle zero-interest-rate period.

For the full 4.2M company universe, see [vivameda.com](https://vivameda.com).
